In [2]:
import json

In [5]:
with open(r"..\assets\ctg-studies.json","r",encoding='utf-8') as f:
    json_obj = json.loads(f.read())
    # f.read()

In [3]:
for item in json_obj:
    for k in item['protocolSection']['eligibilityModule'].keys():
        elibility_criteria = item['protocolSection']['eligibilityModule']
        print(k)
        print(elibility_criteria[k])
    # print(item['protocolSection']['eligibilityModule'])
    break

NameError: name 'json_obj' is not defined

In [6]:
import re

def process_criteria(study_id, raw_text):
    chunks = []
    
    # Split the big block into Inclusion and Exclusion sections
    parts = re.split(r'Exclusion criteria:', raw_text, flags=re.IGNORECASE)
    inclusion_part = parts[0].replace("Inclusion criteria", "").strip()
    exclusion_part = parts[1].strip() if len(parts) > 1 else ""

    # Helper to clean and label bullets
    def clean_bullets(text, label):
        # Find lines starting with numbers or bullets
        bullets = re.findall(r'(?:\d+\.|\*)\s*(.*)', text)
        return [f"{label}: {b.strip()}" for b in bullets if b.strip()]

    chunks.extend(clean_bullets(inclusion_part, "Inclusion Criterion"))
    chunks.extend(clean_bullets(exclusion_part, "Exclusion Criterion"))
    
    return chunks
processed_data = []

for item in json_obj:
    # Get the ID first so every chunk knows where it came from
    nct_id = item['protocolSection']['identificationModule']['nctId']
    eligibility = item['protocolSection']['eligibilityModule']
    raw_criteria = eligibility.get('eligibilityCriteria', '')

    # We split the big string into inclusion and exclusion halves
    parts = re.split(r'Exclusion criteria:', raw_criteria, flags=re.IGNORECASE)
    inclusion_block = parts[0].replace("Inclusion criteria", "").strip()
    exclusion_block = parts[1].strip() if len(parts) > 1 else ""

    # This helper function finds the numbered bullets and adds the label
    def extract_bullets(text, category):
        # We look for lines starting with a number and period or an asterisk
        found = re.findall(r'(?:\d+\.|\*)\s*(.*)', text)
        return [{"text": f"{category} Criterion: {b.strip()}", 
                 "nct_id": nct_id, 
                 "type": category} for b in found if b.strip()]

    # Add the individual labeled bullets to our master list
    processed_data.extend(extract_bullets(inclusion_block, "Inclusion"))
    processed_data.extend(extract_bullets(exclusion_block, "Exclusion"))

# Now you can check the first few to see the structure
print(f"Total chunks created: {len(processed_data)}")
for chunk in processed_data[:3]:
    print(chunk)

Total chunks created: 4282
{'text': 'Inclusion Criterion: Signed study Informed Consent Form.', 'nct_id': 'NCT06549439', 'type': 'Inclusion'}
{'text': 'Inclusion Criterion: Males and females, age ≥ 18 years, no upper age limit.', 'nct_id': 'NCT06549439', 'type': 'Inclusion'}
{'text': 'Inclusion Criterion: Patients with metastatic melanoma and ≥ 1 skin/subcutaneous metastases (clearly definable in clinical examination: largest dimension of ≥ 5mm and ≤ 55 mm; ≤ 2.8 cm thickness (caliper-based measurement); volume ≤ 100ccm) with an indication for palliative radiotherapy of ≥ 1 skin/subcutaneous metastases according to the multidisciplinary tumorboard.', 'nct_id': 'NCT06549439', 'type': 'Inclusion'}


In [7]:
from langchain_core.documents import Document

documents = []

for entry in processed_data:
    # We create the standard LangChain Document object
    doc = Document(
        page_content=entry["text"],
        metadata={
            "nct_id": entry["nct_id"],
            "type": entry["type"],
            "source": "clinicaltrials.gov"
        }
    )
    documents.append(doc)

# Checking our work
print(f"Created {len(documents)} LangChain Document objects.")
print(f"First document content: {documents[0].page_content}")
print(f"First document metadata: {documents[0].metadata}")

Created 4282 LangChain Document objects.
First document content: Inclusion Criterion: Signed study Informed Consent Form.
First document metadata: {'nct_id': 'NCT06549439', 'type': 'Inclusion', 'source': 'clinicaltrials.gov'}


In [ ]:
# You will likely need to: pip install sentence-transformers chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Choose the "Brain" (The Embedding Model)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Build the "Index" (The Vector Store)
vector_db = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings,
    persist_directory="./clinical_trial_index" 
)

print("Vectorization complete. Your index is now saved locally.")

Vectorization complete. Your index is now saved locally.


In [14]:
patient_note = """
Patient presents for follow-up regarding Stage IV metastatic melanoma... 

Clinical Intake Note
Patient: Sarah J.

Age: 54

History of Present Illness: Patient presents for follow-up regarding Stage IV metastatic melanoma. Originally diagnosed with a primary cutaneous lesion on the left shoulder three years ago (Breslow 4.2mm). Underwent wide local excision and sentinel lymph node biopsy at that time.

Last year, imaging confirmed metastasis to the lungs and several subcutaneous sites. She has been on a PD-1 inhibitor (Nivolumab) for six months, but the most recent CT scan shows slight progression in the subcutaneous nodules. She now has two distinct lesions on her upper back, each roughly 2 centimeters in diameter. One of the lesions appears slightly ulcerated.

Past Medical History:

Hypertension (managed with Lisinopril).

History of mild plaque psoriasis (no systemic immunosuppressants required in the last two years).

Appendectomy at age 12.

Current Status:

ECOG Performance Status: 1 (active, but tires easily).

Pregnancy Status: Post-menopausal.

Labs: Normal organ and marrow function; WBC and Platelets within standard range.
"""

# Test: Find the top 5 most relevant rules across all trials
results = vector_db.similarity_search(patient_note, k=5)

for res in results:
    print(f"Trial: {res.metadata['nct_id']}")
    print(f"Rule: {res.page_content}\n")

Trial: NCT06880549
Rule: Inclusion Criterion: Arm 2, Combination with Nivolumab/Relatlimab Arm: Histologically confirmed unresectable or metastatic melanoma, with measurable disease as determined by RECIST v1.1 and have not had any prior therapy for this cancer in this setting (that is, first-line therapy). Note that prior adjuvant, neoadjuvant, or perioperative melanoma therapy (that is, anti-CTLA-4, anti-PD1/L1, BRAF/MEK inhibitors, or interferon) is permitted if disease recurrence did not occur within 3 months from the last treatment date.

Trial: NCT04930783
Rule: Exclusion Criterion: Prior immunotherapy for metastatic melanoma except for anti-CTLA-4. Patients with unresectable melanoma who have received PD-1 inhibition therapy as adjuvant therapy and stopped receiving PD-1 inhibition for a period of ≥ 6 months before starting treatment with Nivolumab or Pembrolizumab are allowed to participate.

Trial: NCT04645680
Rule: Inclusion Criterion: 2 Unresectable Melanoma: i. Histological

In [8]:
import json

with open(r"../assets/fake_patient.json", "r") as f:
    patient_data = json.load(f)

# Extract the diagnosis and current clinical status for the search
clinical_summary = patient_data['clinical_summary']
query_text = (
    f"Patient with {clinical_summary['diagnosis']['condition']}. "
    f"Current status: {clinical_summary['current_status']['new_findings']}. "
    f"Current meds: {clinical_summary['current_status']['current_medication']}."
)
print(query_text)

trial_lookup = {
    item['protocolSection']['identificationModule']['nctId']: 
    item['protocolSection']['eligibilityModule']['eligibilityCriteria']
    for item in json_obj
}

Patient with Stage IV Metastatic Melanoma. Current status: Two nodules on upper back, 2 centimeters each. Current meds: Nivolumab (PD-1 inhibitor) for six months.


In [24]:
from langchain_ollama import OllamaLLM, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Literal
from pydantic import BaseModel, Field
import json

# 1. Define the Structured Output Schema
class TrialEvaluation(BaseModel):
    """The result of evaluating a patient against a clinical trial."""
    nct_id: str = Field(description="The unique identifier for the clinical trial")
    verdict: Literal["MATCH", "MISMATCH", "POTENTIAL"] = Field(description="The final decision")
    match_confidence: int = Field(description="Score from 1-10 on how certain the verdict is")
    reasoning: List[str] = Field(description="Step-by-step logical confirmation points")
    data_gaps: List[str] = Field(description="Missing secondary info like minor labs")
    follow_up_questions: List[str] = Field(description="CRITICAL showstoppers only")

# 2. Initialize both versions of the model
# The "Worker" (Non-chat) - Used for raw clinical reasoning
worker_llm = OllamaLLM(
    model="mistral-nemo", 
    temperature=0, 
    extra_kwargs={"seed": 42}
)

# The "Clerk" (Chat) - Used for structured data extraction
clerk_llm = ChatOllama(
    model="mistral-nemo", 
    temperature=0
).with_structured_output(TrialEvaluation)

# 3. Setup the Prompts
thinking_prompt = ChatPromptTemplate.from_template("""
You are a Senior Clinical Trial Auditor. Use the following hierarchy of logic:

1. TIMELINE CHECK: Is the patient currently on a medication that the rule says must be "Prior" or "Stopped"?
2. CONDITION CHECK: Does the patient's Stage/Diagnosis match the rule?
3. AMBIGUITY CHECK: Is there a specific piece of data (like a date or a lab value) missing that would change the verdict?

PATIENT DATA:
{patient_note}

ELIBILITY RULE:
{trial_rule}

OUTPUT FORMAT:
REASONING: <your step-by-step logic>
VERDICT: <MATCH, MISMATCH, or POTENTIAL>
FOLLOW-UP: <If POTENTIAL, what specific question should we ask the doctor?>
""")

extraction_prompt = ChatPromptTemplate.from_template("""
You are a Data Extraction Agent. Below is a clinical reasoning analysis. 
Your job is to convert this analysis into a structured format without losing the intent.

ANALYSIS:
{analysis}

LOGIC RULES:
- If the analysis mentions a contradiction (e.g. "patient is on Nivolumab but rule says no prior anti-PD1"), VERDICT must be MISMATCH.
- If the analysis says data is missing for a CORE mutation (like BRAF), VERDICT must be POTENTIAL.
- If all core facts align but minor labs are missing, VERDICT must be MATCH.
""")

In [ ]:
# 1. Search the vector DB for relevant trial rules
search_results = vector_db.similarity_search(query_text, k=10)
print(f"Found {len(search_results)} relevant rules. Starting LLM evaluation...\n")

# 2. Get unique Trial IDs
unique_trial_ids = list(set([res.metadata['nct_id'] for res in search_results]))

# 3. Trial lookup dictionary 
trial_lookup = {
    item['protocolSection']['identificationModule']['nctId']: 
    item['protocolSection']['eligibilityModule']['eligibilityCriteria']
    for item in json_obj
}

print(f"Checking {len(unique_trial_ids)} trials in full...\n")

all_results = []

for nct_id in unique_trial_ids:
    full_criteria = trial_lookup.get(nct_id)
    if not full_criteria:
        continue
    
    print(f"--- Processing Trial: {nct_id} ---")
    
    # STAGE 1: Reasoning (The 'Thinking' Step)
    reasoning_chain = thinking_prompt | worker_llm
    raw_analysis = reasoning_chain.invoke({
        "patient_note": json.dumps(clinical_summary, indent=2),
        "trial_rule": full_criteria
    })
    
    # STAGE 2: Extraction (The 'Structuring' Step)
    extraction_chain = extraction_prompt | clerk_llm
    structured_output = extraction_chain.invoke({"analysis": raw_analysis})
    
    # Ensure metadata is correct
    structured_output.nct_id = nct_id
    
    # Store for UI or later use
    all_results.append(structured_output)
    
    # Display results immediately
    print(f"VERDICT: {structured_output.verdict} (Confidence: {structured_output.match_confidence}/100)")
    print(f"REASONING: {structured_output.reasoning[0]}...") 
    if structured_output.follow_up_questions:
        print(f"FOLLOW-UP: {structured_output.follow_up_questions[0]}")
    
    print("\n" + "="*50 + "\n")

Found 10 relevant rules. Starting LLM evaluation...

Checking 8 trials in full...

--- Processing Trial: NCT03021460 ---
VERDICT: MATCH (Confidence: 100/10)
REASONING: TIMELINE_CHECK: The patient is currently on Nivolumab (PD-1 inhibitor), which is not listed in the rule as "Prior" or "Stopped". Therefore, this criterion does not lead to an immediate mismatch....


--- Processing Trial: NCT04930783 ---
VERDICT: POTENTIAL (Confidence: 0/10)
REASONING: TIMELINE CHECK: The patient is currently on Nivolumab for six months. According to the eligibility rule, participants must have recovered from all toxicities associated with prior treatment before starting Nivolumab or Pembrolizumab. However, since this patient has been on Nivolumab for six months without any mention of stopping it, we cannot determine if they have recovered from any potential toxicities....
FOLLOW-UP: Please ask the doctor if the patient has recovered from all toxicities associated with their prior treatment of Nivolumab 

In [23]:
raw_analysis

AIMessage(content="After carefully reviewing the patient's information against the trial criteria, here are the findings:\n\n**Matches:**\n\n1. **Age**: The patient is 54 years old, meeting the age criterion of ≥18 years old.\n2. **ECOG Performance Status**: The patient has an ECOG score of 1, which matches the inclusion criterion of a score of 0-1.\n3. **Histological Confirmation**: The patient has histologically confirmed Stage IV Metastatic Melanoma.\n4. **BRAF Mutation**: While not explicitly stated in the provided information, it's assumed that the patient has a BRAFV600E/K/D/R mutation as they have metastatic melanoma and no mention of wild-type BRAF status.\n5. **Candidacy for SRS Therapy**: This is not explicitly documented but can be assumed based on the presence of intracranial metastases.\n6. **Able to Undergo Gadolinium-Enhanced MRI**: No contraindications are mentioned.\n7. **At Least One Measurable Intracranial Lesion**: The patient has two nodules on the upper back, each